#### Reading posthoc CSV stats
Start 5/5/25. To gather all posthoc csvs exported, unite into one file
v2-5/31/26- add supp figure handling and addition into megafolder 
v3- 
- new null-band percentiles now flow through to the final table as combined Group 1/2 Null Band (5-95%) columns.
- Reorganized supplement panel assignments: added the current supp-CCG names to the panel map and removed stale duplicates.
- Fixed CSV-selection logic so each table contributes its  most-recent file, vs using one global latest date

#### Setup


In [33]:
%pip install -r requirements.txt
from pathlib import Path
import os
import notebook_setup
info = notebook_setup.setup()

# Environment & Imports Setup
import matplotlib as matplotlib
%matplotlib inline 
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from datetime import datetime
import itertools
import ast
##customs 
from helper_functions import *

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


## Preprocess/Gather Files

#### Read and gather all datasets 

In [34]:
#Set/create save and load folder paths 
here = info["repo_root"]
print(f" Here: {here}")
results_location = Path(here).parents[0] / "results"
data_location = Path(here).parents[0] / "data"
print(f"Saving results in {results_location}. Data location is {data_location}")
csv_folder_most_recent = results_location/ f"analysis_CSV_output/" #folders that analysis output goes to
os.chdir(results_location)
print(os.getcwd())


 Here: c:\Users\13car\Dropbox\local_github_repos_personal\dlx56_mPFC_1p_SohalLab\code
Saving results in c:\Users\13car\Dropbox\local_github_repos_personal\dlx56_mPFC_1p_SohalLab\results. Data location is c:\Users\13car\Dropbox\local_github_repos_personal\dlx56_mPFC_1p_SohalLab\data
c:\Users\13car\Dropbox\local_github_repos_personal\dlx56_mPFC_1p_SohalLab\results


In [35]:
folders = []
with os.scandir(os.getcwd()) as dir_iter:
    for item in dir_iter:
        # print(item)
        if item.is_dir():
            folders.append(item)
folders

[<DirEntry 'analysis_CSV_output'>,
 <DirEntry 'CCG_compare_12_Feb_2026 w old_01_21_2026_new_01_31_2026'>,
 <DirEntry 'CCG_compare_12_Feb_2026 w old_01_21_2026_new_02_12_2026'>,
 <DirEntry 'CCG_compare_16_Feb_2026 w old_01_21_2026_new_02_16_2026'>,
 <DirEntry 'CCG_compare_24_Feb_2026 w old_01_21_2026_new_02_16_2026'>,
 <DirEntry 'CCG_compare_25_Feb_2026 w old_01_21_2026_new_02_16_2026'>,
 <DirEntry 'CCG_compare_26_Jan_2026 w old_01_21_2026_new_01_24_2026'>,
 <DirEntry 'CCG_compare_26_Jan_2026 w old_05_31_2025_new_01_23_2026'>,
 <DirEntry 'CCG_compare_26_Jan_2026 w old_05_31_2025_new_01_24_2026'>,
 <DirEntry 'CCG_compare_29_Jan_2026 w old_01_21_2026_new_01_24_2026'>,
 <DirEntry 'CCG_compare_31_Jan_2026 w old_01_21_2026_new_01_31_2026'>,
 <DirEntry 'CCG_comparison_25_Jan_2026'>,
 <DirEntry 'CCG_comparison_26_Jan_2026'>,
 <DirEntry 'date_sorted_figures'>,
 <DirEntry 'dff_baseline_zscore_norm_ensemble_detection_01-Dec-2025_5000 shuffles'>,
 <DirEntry 'dff_baseline_zscore_norm_ensemble_detec

#### Sort folders in directory 

In [36]:
#sort folders in dir by last edited
sorted_folders = sorted(folders, reverse = True, key = lambda entry: entry.stat().st_mtime)
sorted_folders[:10]

[<DirEntry 'figure_tables'>,
 <DirEntry 'analysis_CSV_output'>,
 <DirEntry 'supp_fig_1'>,
 <DirEntry 'supp_fig_2'>,
 <DirEntry 'fig_2'>,
 <DirEntry 'date_sorted_figures'>,
 <DirEntry 'fig_1'>,
 <DirEntry 'supp_fig_4'>,
 <DirEntry 'revision_fig_1'>,
 <DirEntry 'fig_4'>]

In [37]:
sorted_csv_storage = [c for c in sorted_folders if 'analysis_CSV_output' in c.name]
print(sorted_csv_storage)
most_recent_csv_storage = sorted_csv_storage[0]
most_recent_csv_storage

[<DirEntry 'analysis_CSV_output'>]


<DirEntry 'analysis_CSV_output'>

#### get CSV folders in dir

In [38]:
## navigate
os.chdir(most_recent_csv_storage)
folder = os.getcwd()
print(folder)
csv_files = [f for f in os.listdir(folder) if f.lower().endswith(".csv")]
print(f" {len(csv_files)} csv files found in folder")

c:\Users\13car\Dropbox\local_github_repos_personal\dlx56_mPFC_1p_SohalLab\results\analysis_CSV_output
 831 csv files found in folder


In [39]:
csv_files

['1_# Perseverative Errors_behav_posthoc MWU_06_Apr_2026.csv',
 '1_# Perseverative Errors_behav_posthoc MWU_06_May_2026.csv',
 '1_# Perseverative Errors_behav_posthoc MWU_16_Mar_2026.csv',
 '1_# Perseverative Errors_behav_posthoc MWU_18_Jun_2026.csv',
 '1_# Perseverative Errors_behav_posthoc MWU_18_Mar_2026.csv',
 '1_# Perseverative Errors_behav_posthoc MWU_19_Mar_2026.csv',
 '1_# Perseverative Errors_behav_posthoc MWU_24_Feb_2026.csv',
 '1_# Perseverative Errors_behav_posthoc MWU_25_Feb_2026.csv',
 '1_# Perseverative Errors_behav_posthoc MWU_27_Feb_2026.csv',
 '1_# Perseverative Errors_behav_posthoc MWU_28_Feb_2026.csv',
 '1_# Perseverative Errors_behav_posthoc MWU_28_Mar_2026.csv',
 '1_# Perseverative Errors_behav_posthoc MWU_29_Mar_2026.csv',
 '1_behavior_perseverative_error_anova_06_Apr_2026.csv',
 '1_behavior_perseverative_error_anova_06_May_2026.csv',
 '1_behavior_perseverative_error_anova_16_Mar_2026.csv',
 '1_behavior_perseverative_error_anova_18_Jun_2026.csv',
 '1_behavior_per

#### find latest run of data 

In [40]:
def nearest(items, pivot):
    return min(items, key=lambda x: abs(x - pivot)) 
    
def extract_date_str(csv_str, suffix = '.csv', split_delim = '_', get_post_split = -3):
    return csv_str.split(suffix)[0].split(split_delim)[get_post_split:]

In [41]:
date_list = [" ".join(extract_date_str(x)) for x in csv_files ] #drop .csv, then get last 3 entries, but only if has any digit in str
dates_clean = [x for x in date_list if sum(i.isdigit() for i in x) > 5]
dates_clean[-10:]

['23 Feb 2026',
 '25 Feb 2026',
 '26 May 2026',
 '28 Mar 2026',
 '11 May 2026',
 '11 May 2026',
 '04 Jun 2026',
 '09 Jun 2026',
 '11 May 2026',
 '26 May 2026']

In [42]:
now = datetime.today()
print(now)

2026-06-18 14:51:09.737535


#### for each unique fig number, find the csvs with the min timedelta datetime, and collect

In [43]:
def store_csv_name_by_fig_num(csv_files, max_fig_num = 8):
    ''' To- create dict where key = fig num and val = list of csv names with dates in title'''
    num_store = {f: list() for f in range(max_fig_num)}
    ## ## loop throuhg all CSVs
    for f in csv_files:
        fig_n = f.split("_")[0]
        if all((i.isdigit() for i in fig_n)):        # print(fig_n)
            num_store[int(fig_n)].append(f)
    return num_store

In [44]:
num_store = store_csv_name_by_fig_num(csv_files)
num_store

{0: [],
 1: ['1_# Perseverative Errors_behav_posthoc MWU_06_Apr_2026.csv',
  '1_# Perseverative Errors_behav_posthoc MWU_06_May_2026.csv',
  '1_# Perseverative Errors_behav_posthoc MWU_16_Mar_2026.csv',
  '1_# Perseverative Errors_behav_posthoc MWU_18_Jun_2026.csv',
  '1_# Perseverative Errors_behav_posthoc MWU_18_Mar_2026.csv',
  '1_# Perseverative Errors_behav_posthoc MWU_19_Mar_2026.csv',
  '1_# Perseverative Errors_behav_posthoc MWU_24_Feb_2026.csv',
  '1_# Perseverative Errors_behav_posthoc MWU_25_Feb_2026.csv',
  '1_# Perseverative Errors_behav_posthoc MWU_27_Feb_2026.csv',
  '1_# Perseverative Errors_behav_posthoc MWU_28_Feb_2026.csv',
  '1_# Perseverative Errors_behav_posthoc MWU_28_Mar_2026.csv',
  '1_# Perseverative Errors_behav_posthoc MWU_29_Mar_2026.csv',
  '1_behavior_perseverative_error_anova_06_Apr_2026.csv',
  '1_behavior_perseverative_error_anova_06_May_2026.csv',
  '1_behavior_perseverative_error_anova_16_Mar_2026.csv',
  '1_behavior_perseverative_error_anova_18_Jun_

#### create csv store


In [45]:
results_location

WindowsPath('c:/Users/13car/Dropbox/local_github_repos_personal/dlx56_mPFC_1p_SohalLab/results')

In [46]:
csv_store_folder = results_location / "figure_tables"
make_folder(csv_store_folder)

The folder 'c:\Users\13car\Dropbox\local_github_repos_personal\dlx56_mPFC_1p_SohalLab\results\figure_tables' already exists.


## Combine/Save Tables

#### test combination using figure 3 concat:

In [47]:
#
def get_last_fig_csv_names(num_store:dict, fig_num:int, skip_flag = ['anova', 'cells active per trial'],**kwargs):
    ''' For each distinct table in the figure bucket (filename minus its trailing _DD_Mon_YYYY date),
    keep only that table's most-recent dated file. This avoids dropping supplements saved on a
    different date than other tables that share the same fig-number bucket (e.g. many 2_s_* files
    from different notebooks/runs). Returns (current_files, latest_overall_datetime).'''
    fig_storage = num_store[fig_num]
    valid_figs = [x for x in fig_storage if all([skip.lower() not in x.lower() for skip in skip_flag])] #skip_flag: list of str, verify all not present
    latest_by_base = {}  # base name (no date) -> (filename, datetime)
    for f in valid_figs:
        base = "_".join(f.split(".csv")[0].split("_")[:-3])  # strip trailing _DD_Mon_YYYY date tokens
        f_date = datetime.strptime(" ".join(extract_date_str(f)), '%d %b %Y')
        if base not in latest_by_base or f_date > latest_by_base[base][1]:
            latest_by_base[base] = (f, f_date)
    current_files = [fname for fname, _ in latest_by_base.values()]
    closest_time = max((dt for _, dt in latest_by_base.values()), default=datetime.today())
    print(f"fig {fig_num}: {len(current_files)} table(s) selected (latest per table)")
    return current_files, closest_time
    
#get list of dates in csvs for figure of interest, then find closest
def get_latest_csv_datetime(fig_storage:list,skip_flag:list = ["skip"], **kwargs):
    ''' To iterate over list of .csv filenames (with datetime tags embedderd) and find the closest embedded tag to the current time. 
    Skip_flag: list of strings to iterate through and not include in csv list if present'''
    now = datetime.today()
    valid_figs = [x for x in fig_storage if all([skip.lower() not in x.lower() for skip in skip_flag])] #skip_flag: list of str, verify all not present

    date_list_datetime = [datetime.strptime(" ".join(extract_date_str(x)),'%d %b %Y') for x in valid_figs]
    closest_time = min(date_list_datetime , key = lambda x: now- x )
    last_datetime_str = closest_time.strftime('%d_%b_%Y')#convert to str to match save formatting
    print(f"closest datetime: {closest_time}. last_datetime_str = {last_datetime_str}")
    return closest_time, last_datetime_str

def get_fig_csv_matching_datetime(fig_storage, closest_time=datetime.today(), skip_flag:list = ["skip"],):
    ''' To loop over input list of .csv filenames, and pick entries matching the previously found closest datetime'''
    current_files = list()
    #optional- filter list and ignore ones with certain content
    valid_figs = [x for x in fig_storage if all([skip.lower() not in x.lower() for skip in skip_flag])] #skip_flag: list of str, verify all not present
    # print(valid_figs)
    for f in valid_figs:## given closest datetime, iterate and extract 
        f_date = datetime.strptime(" ".join(extract_date_str(f)),'%d %b %Y')# #get string date and convert to datetime to compare against last known date entry
        if f_date == closest_time:
            print(f'matching date file: {f}')
            current_files.append(f)
    
    return current_files


In [48]:
def concat_clean_csv_df(current_files, **kwargs):
    current_dfs = []
    for f in current_files:
        df = pd.read_csv(f)
        df['csv_date'] = " ".join(extract_date_str(os.path.basename(f)))  # e.g. '26 May 2026'
        df['csv_filename'] = os.path.basename(f)  # optional, for full traceability
        current_dfs.append(df)
    combined_fig_df = pd.concat(current_dfs)

    cols_to_drop = ['g_1_child_x', 'g_1_child_y', 'y_is_nonnan', 'y_is_2_elem', 'hue_is_x_axis',
                    'group_1_order_pos', 'group_2_order_pos', 'g_2_child_x', 'g_2_child_y',
                    'tick_text', 'tick_pos', 'hue_group_1_locs', 'hue_group_2_locs', 'Unnamed: 0',
                    'g1_num_loc', 'g2_num_loc', 'g1_cat_loc', 'g2_cat_loc', 'max_group_loc_val', 'plot_name']
    combined_fig_df.drop([c for c in cols_to_drop if c in combined_fig_df.columns], axis=1, inplace=True)
    return combined_fig_df


In [49]:
get_latest_csv_datetime(num_store[1],skip_flag = ['anova', 'trial'])

closest datetime: 2026-06-18 00:00:00. last_datetime_str = 18_Jun_2026


(datetime.datetime(2026, 6, 18, 0, 0), '18_Jun_2026')

In [50]:
fig_num = 3
current_files, closest_time = get_last_fig_csv_names(num_store, fig_num)
combined_fig_df= concat_clean_csv_df(current_files)

combined_fig_df.group_1_n = combined_fig_df.group_1_n.str.replace("(", "", regex = False).str.replace(",)", "", regex=False)
combined_fig_df.group_2_n = combined_fig_df.group_2_n.str.replace("(", "", regex = False).str.replace(",)", "", regex=False)

#key value store for old: new col name
clean_col_name_dict = {'category_compared_within': "Group of posthoc comparison", 
                       'group_1': "Group 1",
                       'group_2': "Group 2", 
                       'group_1_n': "Group 1 N",
                       'group_2_n': "Group 2 N", 
                       'group_1_mean': "Group 1 Mean",
                       'group_1_sem': "Group 1 SEM",
                       'group_2_mean': "Group 2 Mean",
                       'group_2_sem':"Group 2 SEM",
                       'test_name': "Name of Statistical Test",
                       'stat_result': "Test Result",
                       'pvalue': "Test p-value",
                       'categorical_subgroup': "Alternate name- group compared within",
                       'numeric_var': "Variable Compared between Groups",
                       'hue_var': "Variable labeling post-hoc group",
                       'x_category_var': "Categorical variable of plot (x-axis)",
                       'date_tag': "Date of figure creation",
                       'fig_name': "filename of source table",
                       'fig_num': "Figure number"
                      }

combined_fig_df.rename(clean_col_name_dict, axis = 1,inplace = True)
combined_fig_df

fig 3: 28 table(s) selected (latest per table)


,Group of posthoc comparison,Group 1,Group 2,Group 1 N,Group 2 N,Group 1 Mean,Group 1 SEM,Group 2 Mean,Group 2 SEM,Name of Statistical Test,...,Date of figure creation,filename of source table,Figure number,csv_date,csv_filename,group_1_null_low,group_1_null_high,group_2_null_low,group_2_null_high,exceeds_null
0,Early RS Error,WT VEH,Het VEH,10000,10000,94.3869,2.4220,20.0621,3.8066,robust_cohen_d,...,27_Mar_2026,CCG single class pred- Train IA test RS - Earl...,3,27 Mar 2026,3_CCG single class pred- Train IA test RS - Ea...,NaN,NaN,NaN,NaN,NaN
1,Early RS Error,Het VEH,Het CLNZ,10000,10000,20.0621,3.8066,46.2551,2.0052,robust_cohen_d,...,27_Mar_2026,CCG single class pred- Train IA test RS - Earl...,3,27 Mar 2026,3_CCG single class pred- Train IA test RS - Ea...,NaN,NaN,NaN,NaN,NaN
2,Early RS Error,Het VEH,Het postCLNZ,10000,10000,20.0621,3.8066,62.2139,2.7931,robust_cohen_d,...,27_Mar_2026,CCG single class pred- Train IA test RS - Earl...,3,27 Mar 2026,3_CCG single class pred- Train IA test RS - Ea...,NaN,NaN,NaN,NaN,NaN
3,Early RS Correct,WT VEH,Het VEH,10000,10000,54.4617,2.1427,53.7853,2.4082,robust_cohen_d,...,27_Mar_2026,CCG single class pred- Train IA test RS - Earl...,3,27 Mar 2026,3_CCG single class pred- Train IA test RS - Ea...,NaN,NaN,NaN,NaN,NaN
4,Early RS Correct,Het VEH,Het CLNZ,10000,10000,53.7853,2.4082,65.6328,2.4321,robust_cohen_d,...,27_Mar_2026,CCG single class pred- Train IA test RS - Earl...,3,27 Mar 2026,3_CCG single class pred- Train IA test RS - Ea...,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7,Early_RS_Correct,Het VEH,Het CLNZ,10000,10000,0.6688,0.0196,0.5967,0.0149,robust_cohen_d,...,09_Jun_2026,"null_CCGP- Train on IA trials, Test on RS tria...",3,09 Jun 2026,"3_null_CCGP- Train on IA trials, Test on RS tr...",0.409668,0.589844,0.426758,0.574219,True
8,Early_RS_Correct,Het VEH,Het postCLNZ,10000,10000,0.6688,0.0196,0.5647,0.0181,robust_cohen_d,...,09_Jun_2026,"null_CCGP- Train on IA trials, Test on RS tria...",3,09 Jun 2026,"3_null_CCGP- Train on IA trials, Test on RS tr...",0.409668,0.589844,0.399902,0.595703,True
9,Early_RS_Error,WT VEH,Het VEH,10000,10000,0.3523,0.0270,0.3718,0.0158,robust_cohen_d,...,09_Jun_2026,"null_CCGP- Train on IA trials, Test on RS tria...",3,09 Jun 2026,"3_null_CCGP- Train on IA trials, Test on RS tr...",0.388184,0.610352,0.410645,0.587891,True
10,Early_RS_Error,Het VEH,Het CLNZ,10000,10000,0.3718,0.0158,0.6412,0.0168,robust_cohen_d,...,09_Jun_2026,"null_CCGP- Train on IA trials, Test on RS tria...",3,09 Jun 2026,"3_null_CCGP- Train on IA trials, Test on RS tr...",0.410645,0.587891,0.430652,0.569336,True


In [51]:
csv_name = f"fig_{fig_num}_results.csv"
combined_fig_df.to_csv( csv_store_folder/csv_name)

#### Concat posthoc statistics from figures 3-7:

In [52]:
## as figs 1 and 2 don't use traditional posthoc test df outputting, skip that for now 
fig_start = 2
fig_end = 8
##
figs_to_concat = [k for k, v in num_store.items() if v]
print(f" Combining csvs with keys: {figs_to_concat}")
all_fig_tables = []
for fig_num in figs_to_concat:
    current_files, closest_time = get_last_fig_csv_names(num_store, fig_num)
    combined_fig_df= concat_clean_csv_df(current_files)
    combined_fig_df.group_1_n = combined_fig_df.group_1_n.str.replace("(", "", regex = False).str.replace(",)", "", regex=False)
    combined_fig_df.group_2_n = combined_fig_df.group_2_n.str.replace("(", "", regex = False).str.replace(",)", "", regex=False)
    #key value store for old: new col name
    clean_col_name_dict = {'category_compared_within': "Group of posthoc comparison", 
                           'group_1': "Group 1",
                           'group_2': "Group 2", 
                           'group_1_n': "Group 1 N",
                           'group_2_n': "Group 2 N", 
                           'group_1_mean': "Group 1 Mean",
                           'group_1_sem': "Group 1 SEM",
                           'group_2_mean': "Group 2 Mean",
                           'group_2_sem':"Group 2 SEM",
                           'test_name': "Name of Statistical Test",
                           'stat_result': "Test Result",
                           'pvalue': "Test p-value",
                           'categorical_subgroup': "Alternate name- group compared within",
                           'numeric_var': "Variable Compared between Groups",
                           'hue_var': "Variable labeling post-hoc group",
                           'x_category_var': "Categorical variable of plot (x-axis)",
                           'date_tag': "Date of figure creation",
                           'fig_name': "filename of source table",
                           'fig_num': "Figure number"
                          }
    combined_fig_df.rename(clean_col_name_dict, axis = 1,inplace = True)
    all_fig_tables.append(combined_fig_df)
    csv_name = f"fig_{fig_num}_concat_results.csv"
    combined_fig_df.to_csv( csv_store_folder/ csv_name)
full_fig_table = pd.concat(all_fig_tables)


 Combining csvs with keys: [1, 2, 3, 4, 5, 6, 7]
fig 1: 7 table(s) selected (latest per table)
fig 2: 12 table(s) selected (latest per table)
fig 3: 28 table(s) selected (latest per table)
fig 4: 15 table(s) selected (latest per table)
fig 5: 5 table(s) selected (latest per table)
fig 6: 4 table(s) selected (latest per table)
fig 7: 4 table(s) selected (latest per table)


In [53]:
full_fig_table

,Group of posthoc comparison,Group 1,Group 2,Group 1 N,Group 2 N,Group 1 Mean,Group 1 SEM,Group 2 Mean,Group 2 SEM,Name of Statistical Test,...,2.0,3.0,4.0,5.0,group_1_null_low,group_1_null_high,group_2_null_low,group_2_null_high,exceeds_null,comparison
0,geno_day,WT VEH,Het VEH,8,7,3.0000,1.0522,8.1429,0.8845,MWU,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,geno_day,WT VEH,WT CLNZ,8,8,3.0000,1.0522,4.5000,0.5345,MWU,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,geno_day,WT CLNZ,Het VEH,8,7,4.5000,0.5345,8.1429,0.8845,MWU,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,geno_day,Het VEH,Het CLNZ,7,7,8.1429,0.8845,5.1429,0.7997,MWU,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,geno_day,Het VEH,Het postCLNZ,7,7,8.1429,0.8845,5.2857,1.0169,MWU,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1,Late_IA,Het VEH,Het CLNZ,1000,1000,1.1944,0.1677,0.9702,0.1143,robust_cohen_d,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Late_IA_v_Early_RS_Correct
2,Late_IA,Het VEH,Het postCLNZ,1000,1000,1.1944,0.1677,1.0737,0.1479,robust_cohen_d,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Late_IA_v_Early_RS_Correct
3,Early_RS_Correct,WT VEH,Het VEH,1000,1000,0.9893,0.1293,1.1455,0.1412,robust_cohen_d,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Late_IA_v_Early_RS_Correct
4,Early_RS_Correct,Het VEH,Het CLNZ,1000,1000,1.1455,0.1412,1.0002,0.1254,robust_cohen_d,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Late_IA_v_Early_RS_Correct


In [54]:
## save NON STANDARD tables to separate csv for review, then drop from main table
non_standard = full_fig_table[full_fig_table['filename of source table'].isna()]#.dropna(axis=1, how='all')
non_standard.to_csv(csv_store_folder / "non_standard_figure_tables.csv", index=False)
non_standard


,Group of posthoc comparison,Group 1,Group 2,Group 1 N,Group 2 N,Group 1 Mean,Group 1 SEM,Group 2 Mean,Group 2 SEM,Name of Statistical Test,...,2.0,3.0,4.0,5.0,group_1_null_low,group_1_null_high,group_2_null_low,group_2_null_high,exceeds_null,comparison
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.080,0.028,0.009,0.003,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.093,0.032,0.015,0.002,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.075,0.023,0.013,0.004,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.075,0.015,0.004,0.001,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.058,0.012,0.004,0.002,NaN,NaN,NaN,NaN,NaN,NaN


In [55]:
full_fig_table = full_fig_table.dropna(subset=['filename of source table']).dropna(axis=1, how='all')
full_fig_table


,Group of posthoc comparison,Group 1,Group 2,Group 1 N,Group 2 N,Group 1 Mean,Group 1 SEM,Group 2 Mean,Group 2 SEM,Name of Statistical Test,...,filename of source table,Figure number,csv_date,csv_filename,group_1_null_low,group_1_null_high,group_2_null_low,group_2_null_high,exceeds_null,comparison
0,geno_day,WT VEH,Het VEH,8,7,3.0000,1.0522,8.1429,0.8845,MWU,...,# Perseverative Errors,1,18 Jun 2026,1_# Perseverative Errors_behav_posthoc MWU_18_...,NaN,NaN,NaN,NaN,NaN,NaN
1,geno_day,WT VEH,WT CLNZ,8,8,3.0000,1.0522,4.5000,0.5345,MWU,...,# Perseverative Errors,1,18 Jun 2026,1_# Perseverative Errors_behav_posthoc MWU_18_...,NaN,NaN,NaN,NaN,NaN,NaN
2,geno_day,WT CLNZ,Het VEH,8,7,4.5000,0.5345,8.1429,0.8845,MWU,...,# Perseverative Errors,1,18 Jun 2026,1_# Perseverative Errors_behav_posthoc MWU_18_...,NaN,NaN,NaN,NaN,NaN,NaN
3,geno_day,Het VEH,Het CLNZ,7,7,8.1429,0.8845,5.1429,0.7997,MWU,...,# Perseverative Errors,1,18 Jun 2026,1_# Perseverative Errors_behav_posthoc MWU_18_...,NaN,NaN,NaN,NaN,NaN,NaN
4,geno_day,Het VEH,Het postCLNZ,7,7,8.1429,0.8845,5.2857,1.0169,MWU,...,# Perseverative Errors,1,18 Jun 2026,1_# Perseverative Errors_behav_posthoc MWU_18_...,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1,Late_IA,Het VEH,Het CLNZ,1000,1000,1.1944,0.1677,0.9702,0.1143,robust_cohen_d,...,7_H_Autoencoder Late_IA_v_Early_RS_Correct DB ...,7,26 May 2026,7_7_H_Autoencoder Late_IA_v_Early_RS_Correct D...,NaN,NaN,NaN,NaN,NaN,Late_IA_v_Early_RS_Correct
2,Late_IA,Het VEH,Het postCLNZ,1000,1000,1.1944,0.1677,1.0737,0.1479,robust_cohen_d,...,7_H_Autoencoder Late_IA_v_Early_RS_Correct DB ...,7,26 May 2026,7_7_H_Autoencoder Late_IA_v_Early_RS_Correct D...,NaN,NaN,NaN,NaN,NaN,Late_IA_v_Early_RS_Correct
3,Early_RS_Correct,WT VEH,Het VEH,1000,1000,0.9893,0.1293,1.1455,0.1412,robust_cohen_d,...,7_H_Autoencoder Late_IA_v_Early_RS_Correct DB ...,7,26 May 2026,7_7_H_Autoencoder Late_IA_v_Early_RS_Correct D...,NaN,NaN,NaN,NaN,NaN,Late_IA_v_Early_RS_Correct
4,Early_RS_Correct,Het VEH,Het CLNZ,1000,1000,1.1455,0.1412,1.0002,0.1254,robust_cohen_d,...,7_H_Autoencoder Late_IA_v_Early_RS_Correct DB ...,7,26 May 2026,7_7_H_Autoencoder Late_IA_v_Early_RS_Correct D...,NaN,NaN,NaN,NaN,NaN,Late_IA_v_Early_RS_Correct


In [56]:
unique_figs= sorted(full_fig_table['filename of source table'].unique())
# unique_figs

## create dict mapping key (filename) to value (corresponding figure panel)
map_panel_to_filename = {'Autoencoder Early_IA_Correct_v_Early_RS_Correct DB index by ensembles': "6G" ,
 'Autoencoder Early_IA_Correct_v_Late_IA DB index by ensembles': "7D",
 'Autoencoder Early_IA_Error_v_Early_RS_Error DB index by ensembles': "5H" ,
 'Autoencoder Late_IA_v_Early_RS_Correct DB index by ensembles': "7H" ,
 'CCG single class pred- Train Correct test Error- Early_IA_Correct ensemble': "4C" ,
 'CCG single class pred- Train Correct test Error- Early_RS_Error ensemble': "4F" ,
 'CCG single class pred- Train IA test RS - Early_RS_Correct ensemble': "3C" ,
 'Early_IA_Correct_v_Early_RS_Correct SVM accuracy by ensem': "6C" ,
 'Early_IA_Correct_v_Late_IA SVM accuracy by ensem': "7C" ,
 'Early_IA_Error_v_Early_RS_Error SVM accuracy by ensem': "5C" ,
 'Late_IA_v_Early_RS_Correct SVM accuracy by ensem': "7G" ,
 'time-dep decoding - Early RS Correct ens- Early_IA_Correct_v_Early_RS_Correct': "6D" ,
 'time-dep decoding - Early RS Error ens- Early_IA_Error_v_Early_RS_Error': "5D" ,
 # Figure 1 behavior panels (from the Figure 1,2 Plots notebook); combined 'IA_RS division...' left unmapped to avoid duplication
 'IA Performance': "1K" ,
 'RS Performance': "1L" ,
 '# Perseverative Errors': "1M"
                        }
map_supp_panel_to_file = { 'Mean % of cells active in stage that are in stage ensemble':"S1I",
                           'supp_wtclnz_pointplot_Mean event rate by phase':"S1C",
                           'supp_Proportion frames active per trial': "S1B",
 # supp Autoencoder DB-index panels: land on Supplementary Figure 5
    'supp_Autoencoder Early_IA_Correct_v_Early_RS_Correct DB index by ensembles': "S5I" ,
 'supp_Autoencoder Early_IA_Error_v_Early_RS_Error DB index by ensembles': "S5G" ,
 # supp CCGP uniproportion-CCG prediction panels (current '+null' fig_names from SVM v5 cells 76-78): Supplementary Figure 4
 # NOTE: stale 'supp_CCG single class pred-' keys removed -- with per-table CSV selection they
 # would re-surface old-dated files and duplicate the S4C row.
 'supp_CCG+null single class pred- Train IA test RS - Early_RS_Correct ensemble': "S4C",
 'supp_CCG + null 1_class pred-Train_Correct_Test_Error- Early_RS_Error ensemble': "S4E",
 'supp_CCG + null 1_class pred-Train_Correct_Test_Error-Early_IA_Correct ensemble': "S4D"
                         }

legacy_map = {
    'null_CCGP- Train on IA trials, Test on RS trials_Early stage ens_ ': '3B',
    'null CCGP (early ens) train_correct_test_error': '4B',
    'null_CCG uni-class pred- Train IA test RS - Early_RS_Correct ensemble': '3C',
    'null_ CCG uni-class pred- Train Correct test Error- Early_IA_Correct ensemble': '4C',
    'null_CCG uni-class pred- Train Correct test Error- Early_RS_Error ensemble': '4F',
    # 'Supplement- VEH v CLNZ for Het & WT- ensemble proportion overlap': 'S2B',  # moved to the chi-squared Excel sheet (Sheet 2); no longer mapped into Sheet 1
}


full_filename_panel_map = {**map_panel_to_filename, **map_supp_panel_to_file, **legacy_map}
full_filename_panel_map

{'Autoencoder Early_IA_Correct_v_Early_RS_Correct DB index by ensembles': '6G',
 'Autoencoder Early_IA_Correct_v_Late_IA DB index by ensembles': '7D',
 'Autoencoder Early_IA_Error_v_Early_RS_Error DB index by ensembles': '5H',
 'Autoencoder Late_IA_v_Early_RS_Correct DB index by ensembles': '7H',
 'CCG single class pred- Train Correct test Error- Early_IA_Correct ensemble': '4C',
 'CCG single class pred- Train Correct test Error- Early_RS_Error ensemble': '4F',
 'CCG single class pred- Train IA test RS - Early_RS_Correct ensemble': '3C',
 'Early_IA_Correct_v_Early_RS_Correct SVM accuracy by ensem': '6C',
 'Early_IA_Correct_v_Late_IA SVM accuracy by ensem': '7C',
 'Early_IA_Error_v_Early_RS_Error SVM accuracy by ensem': '5C',
 'Late_IA_v_Early_RS_Correct SVM accuracy by ensem': '7G',
 'time-dep decoding - Early RS Correct ens- Early_IA_Correct_v_Early_RS_Correct': '6D',
 'time-dep decoding - Early RS Error ens- Early_IA_Error_v_Early_RS_Error': '5D',
 'IA Performance': '1K',
 'RS Perfor

#### prepare and save concat figure DF

In [57]:
def clean_pvalue_string(x:float): 
    if x < 0.0001:
        if x == 0:
            cleaned =    r"<< 1 x 10^15"
        else:
            cleaned =    f'{x:.2e}'.replace("e", " x 10^")
    else:
        cleaned = f"{x:.4f}"
    return cleaned
import re
panel_re = re.compile(r'^(s_)?(\d+)_?([A-Z]+)_')
## v2 update: to handle combo panels like "5F-H" where multiple letters are present, and to add "S" prefix for supp figs

def extract_panel_id(name): 
    if pd.isna(name):
        return None
    m = panel_re.match(name)
    if not m:
        return None
    supp_prefix = "S" if m.group(1) else ""
    fig_num = m.group(2)
    letters = m.group(3)
    # single letter → "5C"; combo → "5F-H"
    panel = letters if len(letters) == 1 else f"{letters[0]}-{letters[-1]}"
    return f"{supp_prefix}{fig_num}{panel}"


In [58]:
# fresh apply
full_fig_table["Figure Panel"] = full_fig_table['filename of source table'].apply(extract_panel_id)

# definitive view: each unique filename → what panel did it get
view = (full_fig_table.groupby('filename of source table')
        .agg(n_rows=('Figure Panel', 'size'),
             figure_panel=('Figure Panel', 'first')))
print(view)
print(f"\nTotal rows: {len(full_fig_table)}")
print(f"Rows with panel: {full_fig_table['Figure Panel'].notna().sum()}")


                                                    n_rows figure_panel
filename of source table                                               
# Perseverative Errors                                   5         None
5C_SVM classifier accuracy Early_IA_Error_v_Ear...       6           5C
5D_Early RS Error ensemble time-based classific...      18           5D
5D_SVM classifier accuracy Early_IA_Error_v_Ear...       6           5D
5_FGH_Combined_Latent_space_and_DB_index_Early_...       6         5F-H
...                                                    ...          ...
supp_CCG 1_class pred-Train_Correct_Test_Error-...       8         None
supp_CCG single class pred- Train IA test RS - ...       8         None
supp_CCG+null single class pred- Train IA test ...      16         None
supp_Proportion frames active per trial                 48         None
supp_wtclnz_pointplot_Mean event rate by phase          48         None

[68 rows x 2 columns]

Total rows: 780
Rows with panel: 102


In [59]:
# valid_panels = list(full_filename_panel_map.values())
valid_panels = sorted(set(map_panel_to_filename.values())
                     | set(map_supp_panel_to_file.values())
                     | set(legacy_map.values()))

print(f" Keeping rows with panel IDs: {valid_panels}")
# step 1: regex extraction for panel-prefixed filenames (5C_, 7_C_, s_4_C_, etc.)
full_fig_table["Figure Panel"] = full_fig_table['filename of source table'].apply(extract_panel_id)
# step 2: legacy-map fallback for filenames without panel prefix (null_*, supp_*, etc.)
full_fig_table["Figure Panel"] = full_fig_table["Figure Panel"].fillna(
    full_fig_table['filename of source table'].map(full_filename_panel_map)
)
# step 3: keep only rows with a mapped panel (preserves combo tags like 5F-H, 6E-G)
full_fig_table = full_fig_table[full_fig_table["Figure Panel"].notna()]
print(full_fig_table["Figure Panel"].value_counts())
full_fig_table
 
 
# # valid_panels = list(full_filename_panel_map.values())
# valid_panels = sorted(set(map_panel_to_filename.values())
#                      | set(map_supp_panel_to_file.values())
#                      | set(legacy_map.values()))

# print(f" Keeping rows with panel IDs: {valid_panels}")
# full_fig_table["Figure Panel"]= full_fig_table['filename of source table'].replace(full_filename_panel_map)
# full_fig_table = full_fig_table[full_fig_table["Figure Panel"].isin(valid_panels)]## make sure you only keep table with real panels of interest
# full_fig_table


 Keeping rows with panel IDs: ['1K', '1L', '1M', '3B', '3C', '4B', '4C', '4F', '5C', '5D', '5H', '6C', '6D', '6G', '7C', '7D', '7G', '7H', 'S1B', 'S1C', 'S1I', 'S4C', 'S4D', 'S4E', 'S5G', 'S5I']
Figure Panel
S1I     48
S1B     48
S1C     48
5D      24
6D      18
S4E     16
S4D     16
S4C     16
4B      12
4C      12
4F      12
3C      12
3B      12
7D       6
7C       6
7G       6
6G       6
6E-G     6
7H       6
6C       6
5H       6
5F-H     6
5C       6
1K       5
1L       5
1M       5
Name: count, dtype: int64


,Group of posthoc comparison,Group 1,Group 2,Group 1 N,Group 2 N,Group 1 Mean,Group 1 SEM,Group 2 Mean,Group 2 SEM,Name of Statistical Test,...,Figure number,csv_date,csv_filename,group_1_null_low,group_1_null_high,group_2_null_low,group_2_null_high,exceeds_null,comparison,Figure Panel
0,geno_day,WT VEH,Het VEH,8,7,3.0000,1.0522,8.1429,0.8845,MWU,...,1,18 Jun 2026,1_# Perseverative Errors_behav_posthoc MWU_18_...,NaN,NaN,NaN,NaN,NaN,NaN,1M
1,geno_day,WT VEH,WT CLNZ,8,8,3.0000,1.0522,4.5000,0.5345,MWU,...,1,18 Jun 2026,1_# Perseverative Errors_behav_posthoc MWU_18_...,NaN,NaN,NaN,NaN,NaN,NaN,1M
2,geno_day,WT CLNZ,Het VEH,8,7,4.5000,0.5345,8.1429,0.8845,MWU,...,1,18 Jun 2026,1_# Perseverative Errors_behav_posthoc MWU_18_...,NaN,NaN,NaN,NaN,NaN,NaN,1M
3,geno_day,Het VEH,Het CLNZ,7,7,8.1429,0.8845,5.1429,0.7997,MWU,...,1,18 Jun 2026,1_# Perseverative Errors_behav_posthoc MWU_18_...,NaN,NaN,NaN,NaN,NaN,NaN,1M
4,geno_day,Het VEH,Het postCLNZ,7,7,8.1429,0.8845,5.2857,1.0169,MWU,...,1,18 Jun 2026,1_# Perseverative Errors_behav_posthoc MWU_18_...,NaN,NaN,NaN,NaN,NaN,NaN,1M
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1,Late_IA,Het VEH,Het CLNZ,1000,1000,1.1944,0.1677,0.9702,0.1143,robust_cohen_d,...,7,26 May 2026,7_7_H_Autoencoder Late_IA_v_Early_RS_Correct D...,NaN,NaN,NaN,NaN,NaN,Late_IA_v_Early_RS_Correct,7H
2,Late_IA,Het VEH,Het postCLNZ,1000,1000,1.1944,0.1677,1.0737,0.1479,robust_cohen_d,...,7,26 May 2026,7_7_H_Autoencoder Late_IA_v_Early_RS_Correct D...,NaN,NaN,NaN,NaN,NaN,Late_IA_v_Early_RS_Correct,7H
3,Early_RS_Correct,WT VEH,Het VEH,1000,1000,0.9893,0.1293,1.1455,0.1412,robust_cohen_d,...,7,26 May 2026,7_7_H_Autoencoder Late_IA_v_Early_RS_Correct D...,NaN,NaN,NaN,NaN,NaN,Late_IA_v_Early_RS_Correct,7H
4,Early_RS_Correct,Het VEH,Het CLNZ,1000,1000,1.1455,0.1412,1.0002,0.1254,robust_cohen_d,...,7,26 May 2026,7_7_H_Autoencoder Late_IA_v_Early_RS_Correct D...,NaN,NaN,NaN,NaN,NaN,Late_IA_v_Early_RS_Correct,7H


In [60]:

## TEST RELATED PREPROCESS
#Rename varaible names
variable_map = {'prop_active_frames': "% of frames with events", 
                 'mean_rate': "Normalized event rate",
                 'active_in_trial': '% of cells active per trial',
                 'value': '% of samples classified',
                'accuracy':'SVM Accuracy',
                'mean_acc': 'Time-dependent SVM Accuracy',
                'DB_index':'Davies-Bouldin Index of Latent Space activity',
                'IA_TTC': "IA trials to complete",
                'RS_TTC': "RS trials to complete",
                'Perseverative_Error': "# perseverative errors",
               }
full_fig_table['Variable Compared between Groups'] = full_fig_table['Variable Compared between Groups'].map(variable_map)

# robustly parse a stored [statistic, ...] array string into a list of floats.
# Handles numpy reprs (whitespace-separated, width-padded, sci-notation: "[7.78694e+00 0. 2.0e-04]")
# AND python-list reprs (comma-separated: "[1.5444, 0.2139]"). replace(',', ' ') unifies both,
# then split() collapses whitespace runs / drops empties.
def _parse_array_str(x):
    return [float(p) for p in str(x).strip().strip('[]').replace(',', ' ').split()]

#find cohen's d 
cohen_d_mask =  full_fig_table['Name of Statistical Test'] == 'robust_cohen_d'

full_fig_table.loc[cohen_d_mask, 'Test Result'] = full_fig_table.loc[cohen_d_mask, 'Test Result'].apply(_parse_array_str) #apply to cohen's d testing
#clean/redo testing 
test_name_clean = {'robust_cohen_d': "Robust Cohen's d",
                   "permutation_test": "Permutation Test",
                   'MWU': "Mann-Whitney U",
                   'chi_squared':"Chi-Squared"}
full_fig_table['Name of Statistical Test'] = full_fig_table['Name of Statistical Test'].map(test_name_clean) #replace var name with rea lnames 
#for MWU/permutation tests, parse the [statistic, pvalue] array string (same robust parse)
test_stat_spaceless_mask = (full_fig_table['Name of Statistical Test'] == 'Permutation Test') | (full_fig_table['Name of Statistical Test'] == 'Mann-Whitney U')
full_fig_table.loc[test_stat_spaceless_mask, 'Test Result']= full_fig_table.loc[test_stat_spaceless_mask, 'Test Result'].apply(_parse_array_str)
#chi-squared: stat_result is a comma-separated [statistic, pvalue]; parse it too so downstream [0]/round() work
chi_sq_mask = full_fig_table['Name of Statistical Test'] == 'Chi-Squared'
full_fig_table.loc[chi_sq_mask, 'Test Result'] = full_fig_table.loc[chi_sq_mask, 'Test Result'].apply(_parse_array_str)
#clean p-values

full_fig_table['Test Statistic Variable'] = full_fig_table['Name of Statistical Test'].map({"Robust Cohen's d": "Robust Cohen's d",
                                                                                            "Permutation Test": "Mean permutation group diff.",
                                                                                            "Mann-Whitney U Test": "U-statistic",
                                                                                            "Chi-Squared Test": "Chi-Squared"})
full_fig_table['Test Statistic Value'] = full_fig_table['Test Result'].apply(lambda x: x[0])
full_fig_table['Test p-value'] = full_fig_table['Test p-value'].apply(lambda x: clean_pvalue_string(x))
#bugfix- force timebin in comparison string to avoid excel results as dates 
time_dep_rows = full_fig_table['filename of source table'].str.contains("time-dep")
full_fig_table.loc[time_dep_rows,'Group of posthoc comparison']= "timebins: "+ full_fig_table.loc[time_dep_rows,'Group of posthoc comparison']
full_fig_table.loc[time_dep_rows,'Alternate name- group compared within']= "timebins: "+ full_fig_table.loc[time_dep_rows,'Alternate name- group compared within']
full_fig_table.tail()


C:\Users\13car\AppData\Local\Temp\ipykernel_139660\3483444435.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  full_fig_table['Variable Compared between Groups'] = full_fig_table['Variable Compared between Groups'].map(variable_map)
C:\Users\13car\AppData\Local\Temp\ipykernel_139660\3483444435.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  full_fig_table['Name of Statistical Test'] = full_fig_table['Name of Statistical Test'].map(test_name_clean) #replace var name with rea lnames
C:\Users\13car\Ap

,Group of posthoc comparison,Group 1,Group 2,Group 1 N,Group 2 N,Group 1 Mean,Group 1 SEM,Group 2 Mean,Group 2 SEM,Name of Statistical Test,...,csv_filename,group_1_null_low,group_1_null_high,group_2_null_low,group_2_null_high,exceeds_null,comparison,Figure Panel,Test Statistic Variable,Test Statistic Value
1,Late_IA,Het VEH,Het CLNZ,1000,1000,1.1944,0.1677,0.9702,0.1143,Robust Cohen's d,...,7_7_H_Autoencoder Late_IA_v_Early_RS_Correct D...,NaN,NaN,NaN,NaN,NaN,Late_IA_v_Early_RS_Correct,7H,Robust Cohen's d,1.56200
2,Late_IA,Het VEH,Het postCLNZ,1000,1000,1.1944,0.1677,1.0737,0.1479,Robust Cohen's d,...,7_7_H_Autoencoder Late_IA_v_Early_RS_Correct D...,NaN,NaN,NaN,NaN,NaN,Late_IA_v_Early_RS_Correct,7H,Robust Cohen's d,0.76298
3,Early_RS_Correct,WT VEH,Het VEH,1000,1000,0.9893,0.1293,1.1455,0.1412,Robust Cohen's d,...,7_7_H_Autoencoder Late_IA_v_Early_RS_Correct D...,NaN,NaN,NaN,NaN,NaN,Late_IA_v_Early_RS_Correct,7H,Robust Cohen's d,1.15339
4,Early_RS_Correct,Het VEH,Het CLNZ,1000,1000,1.1455,0.1412,1.0002,0.1254,Robust Cohen's d,...,7_7_H_Autoencoder Late_IA_v_Early_RS_Correct D...,NaN,NaN,NaN,NaN,NaN,Late_IA_v_Early_RS_Correct,7H,Robust Cohen's d,1.08773
5,Early_RS_Correct,Het VEH,Het postCLNZ,1000,1000,1.1455,0.1412,0.9668,0.1297,Robust Cohen's d,...,7_7_H_Autoencoder Late_IA_v_Early_RS_Correct D...,NaN,NaN,NaN,NaN,NaN,Late_IA_v_Early_RS_Correct,7H,Robust Cohen's d,1.31770


In [61]:
## update with supplementary table 2 final format 
#reorder then drop figs
new_col_order = ['Figure Panel', 'Group of posthoc comparison', 'Group 1', 'Group 2','Variable Compared between Groups',
                 'Group 1 N', 'Group 2 N', 'Group 1 Mean', 'Group 1 SEM', 'Group 2 Mean', 'Group 2 SEM',
                 'Name of Statistical Test', 'Test p-value','Test Statistic Variable','Test Statistic Value',
                 'Variable labeling post-hoc group',
                 'Categorical variable of plot (x-axis)', 
                 'filename of source table',
                 ]

## final supplement table map
col_rename_map = {'Variable Compared between Groups': 'Dependent Variable',
                  'Name of Statistical Test': 'Statistical Test',
                  'Group of posthoc comparison': 'Posthoc Comparison Group',
                  'Test p-value': 'p-value'}
## carry through null band (5th/95th pct) raw columns when present (CCGP figs 3B/4B)
null_band_raw_cols = ['group_1_null_low', 'group_1_null_high', 'group_2_null_low', 'group_2_null_high']
new_col_order = new_col_order + [c for c in null_band_raw_cols if c in full_fig_table.columns]
full_fig_table = full_fig_table.loc[:, new_col_order].rename(columns = col_rename_map)
## combine 'Group 1' / 'Group 2' name columns into a single 'Group 1 & 2' column (joined with ' vs. '), in place
full_fig_table.insert(full_fig_table.columns.get_loc('Group 1'), 'Group 1 & 2',
                      full_fig_table.apply(lambda x: f"{x['Group 1']}-{x['Group 2']}", axis=1))
full_fig_table = full_fig_table.drop(columns=['Group 1', 'Group 2'])
full_fig_table['shorthand test name'] = full_fig_table['Statistical Test'].map({"Robust Cohen's d": "d","Mann-Whitney U": "U","Chi-Squared": "χ2"})
full_fig_table['Test stat., Name, Value']= full_fig_table.apply(lambda x: f'{x['shorthand test name']}={round(x["Test Statistic Value"],3)}', axis=1)
# full_fig_table['Grouping Variable ']= full_fig_table['Variable Compared between Groups']

full_fig_table

,Figure Panel,Posthoc Comparison Group,Group 1 & 2,Dependent Variable,Group 1 N,Group 2 N,Group 1 Mean,Group 1 SEM,Group 2 Mean,Group 2 SEM,...,Test Statistic Value,Variable labeling post-hoc group,Categorical variable of plot (x-axis),filename of source table,group_1_null_low,group_1_null_high,group_2_null_low,group_2_null_high,shorthand test name,"Test stat., Name, Value"
0,1M,geno_day,WT VEH-Het VEH,# perseverative errors,8,7,3.0000,1.0522,8.1429,0.8845,...,6.00000,geno_day,geno_day,# Perseverative Errors,NaN,NaN,NaN,NaN,U,U=6.0
1,1M,geno_day,WT VEH-WT CLNZ,# perseverative errors,8,8,3.0000,1.0522,4.5000,0.5345,...,13.00000,geno_day,geno_day,# Perseverative Errors,NaN,NaN,NaN,NaN,U,U=13.0
2,1M,geno_day,WT CLNZ-Het VEH,# perseverative errors,8,7,4.5000,0.5345,8.1429,0.8845,...,5.00000,geno_day,geno_day,# Perseverative Errors,NaN,NaN,NaN,NaN,U,U=5.0
3,1M,geno_day,Het VEH-Het CLNZ,# perseverative errors,7,7,8.1429,0.8845,5.1429,0.7997,...,41.00000,geno_day,geno_day,# Perseverative Errors,NaN,NaN,NaN,NaN,U,U=41.0
4,1M,geno_day,Het VEH-Het postCLNZ,# perseverative errors,7,7,8.1429,0.8845,5.2857,1.0169,...,38.50000,geno_day,geno_day,# Perseverative Errors,NaN,NaN,NaN,NaN,U,U=38.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1,7H,Late_IA,Het VEH-Het CLNZ,Davies-Bouldin Index of Latent Space activity,1000,1000,1.1944,0.1677,0.9702,0.1143,...,1.56200,geno_day,ensemble,7_H_Autoencoder Late_IA_v_Early_RS_Correct DB ...,NaN,NaN,NaN,NaN,d,d=1.562
2,7H,Late_IA,Het VEH-Het postCLNZ,Davies-Bouldin Index of Latent Space activity,1000,1000,1.1944,0.1677,1.0737,0.1479,...,0.76298,geno_day,ensemble,7_H_Autoencoder Late_IA_v_Early_RS_Correct DB ...,NaN,NaN,NaN,NaN,d,d=0.763
3,7H,Early_RS_Correct,WT VEH-Het VEH,Davies-Bouldin Index of Latent Space activity,1000,1000,0.9893,0.1293,1.1455,0.1412,...,1.15339,geno_day,ensemble,7_H_Autoencoder Late_IA_v_Early_RS_Correct DB ...,NaN,NaN,NaN,NaN,d,d=1.153
4,7H,Early_RS_Correct,Het VEH-Het CLNZ,Davies-Bouldin Index of Latent Space activity,1000,1000,1.1455,0.1412,1.0002,0.1254,...,1.08773,geno_day,ensemble,7_H_Autoencoder Late_IA_v_Early_RS_Correct DB ...,NaN,NaN,NaN,NaN,d,d=1.088


In [62]:
## cleaning up posthoc comparisons
full_fig_table['Posthoc Comparison Group'] = full_fig_table['Posthoc Comparison Group'].str.replace("_", " ")
## time-dependent decoding clean up
time_dep_rows = full_fig_table['Posthoc Comparison Group'].str.contains("-")
full_fig_table.loc[time_dep_rows,'Posthoc Comparison Group'] = full_fig_table.loc[time_dep_rows,'Posthoc Comparison Group']+ " seconds from outcome"


In [63]:
## v2 update- combine columns for space
#  create group 1, 2 N column
full_fig_table['N- Group 1 & 2 '] = full_fig_table.apply(lambda x: f"{x['Group 1 N']}, {x['Group 2 N']}", axis=1)
#  create +/- sem col
full_fig_table['Group 1 Mean +/- SEM'] = (full_fig_table['Group 1 Mean'].map('{:.3f}'.format)
    + ' +/- '  + full_fig_table['Group 1 SEM'].map('{:.3f}'.format))
full_fig_table['Group 2 Mean +/- SEM'] = (full_fig_table['Group 2 Mean'].map('{:.3f}'.format)
    + ' +/- ' + full_fig_table['Group 2 SEM'].map('{:.3f}'.format))
## null band (5th-95th percentile) combined into one string per group, like Mean +/- SEM
null_band_pairs = {'Group 1 Null Band (5-95%)': ('group_1_null_low', 'group_1_null_high'),
                   'Group 2 Null Band (5-95%)': ('group_2_null_low', 'group_2_null_high')}
for band_col, (lo_col, hi_col) in null_band_pairs.items():
    if lo_col in full_fig_table.columns and hi_col in full_fig_table.columns:
        full_fig_table[band_col] = full_fig_table.apply(
            lambda x, lo=lo_col, hi=hi_col: f"{x[lo]:.3f} - {x[hi]:.3f}" if pd.notna(x[lo]) and pd.notna(x[hi]) else "",
            axis=1)
full_fig_table.head()

,Figure Panel,Posthoc Comparison Group,Group 1 & 2,Dependent Variable,Group 1 N,Group 2 N,Group 1 Mean,Group 1 SEM,Group 2 Mean,Group 2 SEM,...,group_1_null_high,group_2_null_low,group_2_null_high,shorthand test name,"Test stat., Name, Value",N- Group 1 & 2,Group 1 Mean +/- SEM,Group 2 Mean +/- SEM,Group 1 Null Band (5-95%),Group 2 Null Band (5-95%)
0,1M,geno day,WT VEH-Het VEH,# perseverative errors,8,7,3.0000,1.0522,8.1429,0.8845,...,NaN,NaN,NaN,U,U=6.0,"8, 7",3.000 +/- 1.052,8.143 +/- 0.884,,
1,1M,geno day,WT VEH-WT CLNZ,# perseverative errors,8,8,3.0000,1.0522,4.5000,0.5345,...,NaN,NaN,NaN,U,U=13.0,"8, 8",3.000 +/- 1.052,4.500 +/- 0.534,,
2,1M,geno day,WT CLNZ-Het VEH,# perseverative errors,8,7,4.5000,0.5345,8.1429,0.8845,...,NaN,NaN,NaN,U,U=5.0,"8, 7",4.500 +/- 0.534,8.143 +/- 0.884,,
3,1M,geno day,Het VEH-Het CLNZ,# perseverative errors,7,7,8.1429,0.8845,5.1429,0.7997,...,NaN,NaN,NaN,U,U=41.0,"7, 7",8.143 +/- 0.884,5.143 +/- 0.800,,
4,1M,geno day,Het VEH-Het postCLNZ,# perseverative errors,7,7,8.1429,0.8845,5.2857,1.0169,...,NaN,NaN,NaN,U,U=38.5,"7, 7",8.143 +/- 0.884,5.286 +/- 1.017,,


#### Save full figure table after concats 

In [64]:
#save fig table
cols_drop_in_save = ['Test Result', 
                     'Alternate name- group compared within', 
                     'comparison',
                     'Figure number',
                     'Date of figure creation',
                     'Variable labeling post-hoc group',
                     'Categorical variable of plot (x-axis)',
                     'shorthand test name', 
                     'Test Statistic Value',
                     'Test Statistic Variable',
                     'Group 1 N',
                    'Group 2 N',
                     'Group 1 Mean', 
                     'Group 1 SEM', 
                     'Group 2 Mean',
                    'Group 2 SEM',
                     'group_1_null_low', 'group_1_null_high',
                     'group_2_null_low', 'group_2_null_high'] ## drop any columns in this list that are still present in the table before saving, to avoid saving extraneous info

full_fig_table = full_fig_table.drop([c for c in cols_drop_in_save if c in full_fig_table.columns],axis = 1)
csv_name = f"all_figure_concat_results.csv"
full_fig_table.set_index("Figure Panel").to_csv( csv_store_folder/csv_name)

## ---- unified chi-squared ensemble-overlap sheet: geno table + S2B, deduped at the genotype-pair level ----
def _latest_by_date(files):
    return max(files, key=lambda f: datetime.strptime(" ".join(extract_date_str(f)), '%d %b %Y'))

chi2_sheet = None
geno_files = [f for f in os.listdir(csv_folder_most_recent) if f.startswith("ensemble_overlap_by_geno_")]
if geno_files:
    # normalize column names so this works whether the geno table is snake_case or Title Cased
    geno = pd.read_csv(csv_folder_most_recent / _latest_by_date(geno_files))
    geno.columns = [str(c).strip().lower().replace(' ', '_') for c in geno.columns]
    geno = geno.drop(columns=[c for c in ['unnamed:_0', 'geno_compare_overlap', 'expected_frequency'] if c in geno.columns])

    s2b_files = [f for f in os.listdir(csv_folder_most_recent)
                 if "Supplement- VEH v CLNZ for Het & WT- ensemble proportion overlap" in f and f.lower().endswith(".csv")]
    s2b_geno_pairs = set()
    if s2b_files:
        s2b = pd.read_csv(csv_folder_most_recent / _latest_by_date(s2b_files))
        s2b_geno_pairs = set(zip(s2b['group_1'], s2b['group_2']))
        geno_pairs_present = set(zip(geno['geno_day_geno_1'], geno['geno_day_geno_2']))
        # append only S2B rows whose genotype-pair isn't already in the geno table (dedup by genotype-pair)
        s2b_uniq = s2b[~s2b.apply(lambda r: (r['group_1'], r['group_2']) in geno_pairs_present, axis=1)]
        parts = s2b_uniq['category_compared_within'].str.split('_x_', expand=True)
        s2b_re = pd.DataFrame({
            'geno_day_geno_1': s2b_uniq['group_1'],
            'geno_day_geno_2': s2b_uniq['group_2'],
            'geno_comparison': s2b_uniq['group_1'] + ' vs ' + s2b_uniq['group_2'],
            'ensemble_comparison': s2b_uniq['category_compared_within'].str.replace('_x_', ' vs ', regex=False).str.replace('_', ' ', regex=False),
            'set_1_name': parts[0].str.replace('_', ' ', regex=False),
            'set_2_name': parts[1].str.replace('_', ' ', regex=False),
            'statistic': s2b_uniq['stat_result'].apply(lambda v: float(str(v).strip('[]').split(',')[0])),
            'pvalue': s2b_uniq['pvalue'],
        })
        geno = pd.concat([geno, s2b_re], ignore_index=True)
    # keep the panel reference: tag the comparisons that belong to the S2B panel
    geno['Figure Panel'] = geno.apply(lambda r: 'S2B' if (r['geno_day_geno_1'], r['geno_day_geno_2']) in s2b_geno_pairs else '', axis=1)
    col_order = ['Figure Panel', 'geno_comparison', 'geno_day_geno_1', 'geno_day_geno_2', 'ensemble_comparison',
                 'set_1_name', 'set_2_name', 'total_overlap_geno_1', 'n_total_geno_1', 'total_overlap_geno_2',
                 'n_total_geno_2', 'statistic', 'pvalue', 'dof']
    chi2_sheet = geno[[c for c in col_order if c in geno.columns]].copy()
    chi2_sheet['row_type'] = 'geno-vs-geno overlap chi2'

    ## ---- fuse in overlap_tab (per-geno venn-diagram overlap), tagged with its venn panel ----
    ot_files = [f for f in os.listdir(csv_folder_most_recent) if f.startswith("ensemble_overlap_tab_")]
    if ot_files:
        ot = pd.read_csv(csv_folder_most_recent / _latest_by_date(ot_files))
        ot.columns = [str(c).strip().lower().replace(' ', '_') for c in ot.columns]
        ot = ot.drop(columns=[c for c in ['unnamed:_0', 'contingency', 'expected_frequency'] if c in ot.columns])
        # each ensemble pair -> its venn-diagram panel
        venn_panel = {('Early_IA_Error', 'Early_RS_Error'): '5A',
                      ('Early_IA_Correct', 'Early_RS_Correct'): '6A',
                      ('Early_IA_Correct', 'Late_IA'): '7A',
                      ('Late_IA', 'Early_RS_Correct'): '7E'}
        _ns = lambda s: str(s).strip().replace(' ', '_')   # normalize set name (snake_case or Title Cased)
        ot['Figure Panel'] = ot.apply(lambda r: venn_panel.get((_ns(r['set_1_name']), _ns(r['set_2_name'])), ''), axis=1)
        ot['row_type'] = 'per-geno venn overlap'
        chi2_sheet = pd.concat([chi2_sheet, ot], ignore_index=True)
    else:
        print("note: no ensemble_overlap_tab_* file found -- venn overlap_tab not fused (re-run Ensembles notebook)")

    # final column order: lead with panel + type, then identity, then counts/stats
    lead = ['Figure Panel', 'row_type', 'geno_comparison', 'geno_day', 'geno_day_geno_1', 'geno_day_geno_2',
            'ensemble_comparison', 'set_1_name', 'set_2_name']
    chi2_sheet = chi2_sheet[[c for c in lead if c in chi2_sheet.columns] + [c for c in chi2_sheet.columns if c not in lead]]
    print(f"chi-squared/overlap sheet: {len(chi2_sheet)} rows | panels: {sorted(set(chi2_sheet['Figure Panel']) - {''})}")
else:
    print("WARNING: no ensemble_overlap_by_geno_* file found -- chi-squared sheet skipped")

xlsx_name = "all_figure_concat_results.xlsx"
with pd.ExcelWriter(csv_store_folder / xlsx_name) as writer:
    full_fig_table.set_index("Figure Panel").to_excel(writer, sheet_name="all figure concat results")
    if chi2_sheet is not None:
        chi2_sheet.to_excel(writer, sheet_name="geno ensemble overlap chi2", index=False)
full_fig_table

chi-squared/overlap sheet: 36 rows | panels: ['5A', '6A', '7A', '7E', 'S2B']


,Figure Panel,Posthoc Comparison Group,Group 1 & 2,Dependent Variable,Statistical Test,p-value,filename of source table,"Test stat., Name, Value",N- Group 1 & 2,Group 1 Mean +/- SEM,Group 2 Mean +/- SEM,Group 1 Null Band (5-95%),Group 2 Null Band (5-95%)
0,1M,geno day,WT VEH-Het VEH,# perseverative errors,Mann-Whitney U,0.0117,# Perseverative Errors,U=6.0,"8, 7",3.000 +/- 1.052,8.143 +/- 0.884,,
1,1M,geno day,WT VEH-WT CLNZ,# perseverative errors,Mann-Whitney U,0.0471,# Perseverative Errors,U=13.0,"8, 8",3.000 +/- 1.052,4.500 +/- 0.534,,
2,1M,geno day,WT CLNZ-Het VEH,# perseverative errors,Mann-Whitney U,0.0082,# Perseverative Errors,U=5.0,"8, 7",4.500 +/- 0.534,8.143 +/- 0.884,,
3,1M,geno day,Het VEH-Het CLNZ,# perseverative errors,Mann-Whitney U,0.0387,# Perseverative Errors,U=41.0,"7, 7",8.143 +/- 0.884,5.143 +/- 0.800,,
4,1M,geno day,Het VEH-Het postCLNZ,# perseverative errors,Mann-Whitney U,0.0784,# Perseverative Errors,U=38.5,"7, 7",8.143 +/- 0.884,5.286 +/- 1.017,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1,7H,Late IA,Het VEH-Het CLNZ,Davies-Bouldin Index of Latent Space activity,Robust Cohen's d,0.0591,7_H_Autoencoder Late_IA_v_Early_RS_Correct DB ...,d=1.562,"1000, 1000",1.194 +/- 0.168,0.970 +/- 0.114,,
2,7H,Late IA,Het VEH-Het postCLNZ,Davies-Bouldin Index of Latent Space activity,Robust Cohen's d,0.2227,7_H_Autoencoder Late_IA_v_Early_RS_Correct DB ...,d=0.763,"1000, 1000",1.194 +/- 0.168,1.074 +/- 0.148,,
3,7H,Early RS Correct,WT VEH-Het VEH,Davies-Bouldin Index of Latent Space activity,Robust Cohen's d,0.1244,7_H_Autoencoder Late_IA_v_Early_RS_Correct DB ...,d=1.153,"1000, 1000",0.989 +/- 0.129,1.145 +/- 0.141,,
4,7H,Early RS Correct,Het VEH-Het CLNZ,Davies-Bouldin Index of Latent Space activity,Robust Cohen's d,0.1384,7_H_Autoencoder Late_IA_v_Early_RS_Correct DB ...,d=1.088,"1000, 1000",1.145 +/- 0.141,1.000 +/- 0.125,,


In [65]:
full_fig_table['Figure Panel'].value_counts()

Figure Panel
S1I     48
S1B     48
S1C     48
5D      24
6D      18
S4E     16
S4D     16
S4C     16
4B      12
4C      12
4F      12
3C      12
3B      12
7D       6
7C       6
7G       6
6G       6
6E-G     6
7H       6
6C       6
5H       6
5F-H     6
5C       6
1K       5
1L       5
1M       5
Name: count, dtype: int64